# Condition Visualization Viewer

Interactive exploration of how learned conditions affect retrieval.
Load the saved snapshots from training and apply any condition you like.

**Data source**: `<experiment_dir>/condition_viz/`

- `fixed_data.pt` — frozen CLIP embeddings + raw data (saved once, backbone never changes)
- `epoch_XXXX.pt` — `label_embeddings_all`, `representatives`, `combiner_state_dict`, `combiner_config`


In [1]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
EXPERIMENT_DIR = (
    "/project/CoSiR/res/CoSiR_Experiment/redcaps2/20260421_144100_CoSiR_Experiment"
)

EPOCH = None  # None → load the latest available epoch
DEVICE = "cuda"  # "cpu" if no GPU available

N_RETRIEVE = 5  # top-K results to show per query
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
import os, sys, glob, textwrap
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

sys.path.insert(
    0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "../../..")
)
from src.model.combiner import Combiner_new

cond_dir = os.path.join(EXPERIMENT_DIR, "condition_viz")

# ── Fixed data (backbone embeddings, never change) ─────────────────────────
fixed = torch.load(os.path.join(cond_dir, "fixed_data.pt"), map_location="cpu")
all_img_emb = fixed["all_img_emb"]  # [N_img, D]
all_txt_emb = fixed["all_txt_emb"]  # [N_txt, D]
all_raw_text = fixed["all_raw_text"]  # list[str]
image_paths = fixed["image_paths"]  # list[str]
image_to_text_map = fixed["image_to_text_map"]  # [N_img, cpi]
text_to_image_map = fixed["text_to_image_map"]  # [N_txt]
cpi = fixed["captions_per_image"]
n_img, n_txt = all_img_emb.shape[0], all_txt_emb.shape[0]
print(f"Fixed data: {n_img} images, {n_txt} texts, {cpi} captions/image")

# ── Epoch snapshot ─────────────────────────────────────────────────────────
epoch_files = sorted(glob.glob(os.path.join(cond_dir, "epoch_*.pt")))
if not epoch_files:
    raise FileNotFoundError(f"No epoch snapshots in {cond_dir}")
snap_path = (
    epoch_files[-1]
    if EPOCH is None
    else os.path.join(cond_dir, f"epoch_{EPOCH:04d}.pt")
)
snap = torch.load(snap_path, map_location="cpu", weights_only=False)
epoch = snap["epoch"]
label_embeddings_all = snap["label_embeddings_all"]  # [N_samples, label_dim]
representatives = snap["representatives"]  # [K, label_dim]
combiner_cfg = snap["combiner_config"]
K = representatives.shape[0]
print(
    f"Epoch {epoch} | {label_embeddings_all.shape[0]} conditions | {K} representatives"
)

# ── Combiner ───────────────────────────────────────────────────────────────
combiner = Combiner_new(**combiner_cfg).to(DEVICE)
combiner.load_state_dict(snap["combiner_state_dict"])
combiner.eval()
print("Combiner loaded.")

## (Optional) Remap image paths

If the experiment was run on a different machine, adjust the path prefix below.


In [ ]:
# Uncomment and edit if image paths need remapping (different machine)
# old_prefix = "/var/scratch/wding/Dataset/redcaps_test/"
# new_prefix = "/data/PDD/"
# image_paths = [p.replace(old_prefix, new_prefix) for p in image_paths]
print(image_paths[0])

## Precompute — run once

Runs the combiner for all K representatives + CLIP baseline, stores top-K indices in CPU memory.
After this cell no GPU is needed for any visualization.


In [ ]:
PRECOMP_K = N_RETRIEVE  # top-K to store per query

img_n = F.normalize(all_img_emb.to(DEVICE), dim=-1)  # [N_img, D]
txt_raw = all_txt_emb.to(DEVICE)  # [N_txt, D]
txt_n = F.normalize(txt_raw, dim=-1)  # for CLIP baseline

# Storage: [K, N_img, PRECOMP_K] and [K, N_txt, PRECOMP_K]
pc_i2t = torch.zeros(
    K, n_img, PRECOMP_K, dtype=torch.long
)  # top-K text indices per image query
pc_t2i = torch.zeros(
    K, n_txt, PRECOMP_K, dtype=torch.long
)  # top-K image indices per text query

with torch.no_grad():
    for k_idx in range(K):
        cond = representatives[k_idx].to(DEVICE)

        # I2T: modulate all texts, then score against all image queries
        txt_mod = combiner(
            txt_raw, None, cond.unsqueeze(0).expand(n_txt, -1)
        )  # [N_txt, D]
        sim_i2t = img_n @ txt_mod.T  # [N_img, N_txt]
        pc_i2t[k_idx] = torch.topk(sim_i2t, k=PRECOMP_K, dim=1).indices.cpu()

        # T2I: modulate each text query, then score against all images
        # Process in batches to avoid OOM on large datasets
        batch = 512
        rows = []
        for start in range(0, n_txt, batch):
            end = min(start + batch, n_txt)
            cond_b = cond.unsqueeze(0).expand(end - start, -1)
            txt_mod_b = combiner(txt_raw[start:end], None, cond_b)  # [B, D]
            rows.append(
                torch.topk(txt_mod_b @ img_n.T, k=PRECOMP_K, dim=1).indices.cpu()
            )
        pc_t2i[k_idx] = torch.cat(rows, dim=0)

        print(f"  rep {k_idx+1}/{K} done", end="\r")

# CLIP baseline (no condition)
sim_clip_i2t = (img_n @ txt_n.T).cpu()  # [N_img, N_txt]
sim_clip_t2i = (txt_n @ img_n.T).cpu()  # [N_txt, N_img]
clip_i2t = torch.topk(sim_clip_i2t, k=PRECOMP_K, dim=1).indices  # [N_img, PRECOMP_K]
clip_t2i = torch.topk(sim_clip_t2i, k=PRECOMP_K, dim=1).indices  # [N_txt, PRECOMP_K]

del img_n, txt_raw, txt_n
torch.cuda.empty_cache()
print(
    f"\nDone. Precomputed {K} representatives × {n_img} image queries × {n_txt} text queries."
)

## Visualization helpers


In [ ]:
def _gt_i2t(query_img_idx, top_k_list):
    gt = set(image_to_text_map[query_img_idx].tolist())
    return [t in gt for t in top_k_list]


def _gt_t2i(query_txt_idx, top_k_list):
    gt = text_to_image_map[query_txt_idx].item()
    return [i == gt for i in top_k_list]


def _short_cap(tidx, max_words=20):
    cap = all_raw_text[tidx] if tidx < len(all_raw_text) else "?"
    words = cap.split()
    return (" ".join(words[:max_words]) + "…") if len(words) > max_words else cap


def _img(idx):
    try:
        return Image.open(image_paths[idx]).convert("RGB")
    except Exception:
        return None


def show_i2t(query_img_idx: int, rep_indices=None, k: int = PRECOMP_K):
    """Image → text: query image on the left, CLIP baseline + one column per representative."""
    reps = list(range(K)) if rep_indices is None else rep_indices
    columns = [
        {
            "label": "CLIP\n(baseline)",
            "top_k": clip_i2t[query_img_idx, :k].tolist(),
            "is_gt": _gt_i2t(query_img_idx, clip_i2t[query_img_idx, :k].tolist()),
        }
    ]
    for r in reps:
        top = pc_i2t[r, query_img_idx, :k].tolist()
        columns.append(
            {"label": f"rep_{r}", "top_k": top, "is_gt": _gt_i2t(query_img_idx, top)}
        )

    img_w, col_w, row_h = 2.2, 3.0, 1.0
    fig = plt.figure(
        figsize=(img_w + col_w * len(columns), row_h * k + 0.5), constrained_layout=True
    )
    gs = gridspec.GridSpec(
        k, 1 + len(columns), figure=fig, width_ratios=[img_w] + [col_w] * len(columns)
    )

    ax0 = fig.add_subplot(gs[:, 0])
    img = _img(query_img_idx)
    (
        ax0.imshow(img)
        if img
        else ax0.text(
            0.5,
            0.5,
            f"img {query_img_idx}",
            ha="center",
            va="center",
            transform=ax0.transAxes,
            fontsize=7,
        )
    )
    ax0.set_title(f"Query\nimg {query_img_idx}", fontsize=8, pad=2)
    ax0.axis("off")

    for col, cd in enumerate(columns):
        for row in range(k):
            ax = fig.add_subplot(gs[row, col + 1])
            tidx = cd["top_k"][row]
            gt = cd["is_gt"][row]
            bg = "#b6f5b0" if gt else "white"
            ax.set_facecolor(bg)
            ax.text(
                0.5,
                0.5,
                f"#{row+1}  {'✓' if gt else ''}\n"
                + "\n".join(textwrap.wrap(_short_cap(tidx), 28)),
                ha="center",
                va="center",
                fontsize=6,
                transform=ax.transAxes,
                bbox=dict(boxstyle="square,pad=0", facecolor=bg, linewidth=0),
            )
            ax.axis("off")
            if row == 0:
                ax.set_title(cd["label"], fontsize=7, pad=2)

    fig.suptitle(f"I2T  image {query_img_idx}  [epoch {epoch}]", fontsize=9)
    plt.show()


def show_t2i(query_txt_idx: int, rep_indices=None, k: int = PRECOMP_K):
    """Text → image: query caption at top, CLIP baseline + one column per representative."""
    reps = list(range(K)) if rep_indices is None else rep_indices
    query_cap = _short_cap(query_txt_idx, max_words=30)
    gt_img = text_to_image_map[query_txt_idx].item()

    columns = [
        {
            "label": "CLIP\n(baseline)",
            "top_k": clip_t2i[query_txt_idx, :k].tolist(),
            "is_gt": _gt_t2i(query_txt_idx, clip_t2i[query_txt_idx, :k].tolist()),
        }
    ]
    for r in reps:
        top = pc_t2i[r, query_txt_idx, :k].tolist()
        columns.append(
            {"label": f"rep_{r}", "top_k": top, "is_gt": _gt_t2i(query_txt_idx, top)}
        )

    img_w, img_h = 1.8, 1.5
    n_cols = len(columns)
    q_short = "\n".join(textwrap.wrap(query_cap, width=80))
    n_caption_lines = len(q_short.split("\n")) + 1
    top_margin = 0.4 + 0.22 * n_caption_lines

    fig, axes = plt.subplots(
        k, n_cols, figsize=(img_w * n_cols, img_h * k + top_margin), squeeze=False
    )
    fig.subplots_adjust(top=1 - top_margin / (img_h * k + top_margin))

    for col, cd in enumerate(columns):
        for row in range(k):
            ax = axes[row, col]
            iidx = cd["top_k"][row]
            gt = cd["is_gt"][row]
            img = _img(iidx)
            if img:
                ax.imshow(img)
            else:
                ax.text(
                    0.5,
                    0.5,
                    f"img {iidx}",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    fontsize=7,
                )
            ax.set_xlabel(f"#{row+1} img{iidx}{'  ✓' if gt else ''}", fontsize=6)
            if gt:
                for spine in ax.spines.values():
                    spine.set_edgecolor("limegreen")
                    spine.set_linewidth(3)
            ax.set_xticks([])
            ax.set_yticks([])
            if row == 0:
                ax.set_title(cd["label"], fontsize=7, pad=2)

    fig.suptitle(
        f'T2I  text {query_txt_idx}  GT img {gt_img}  [epoch {epoch}]\n"{q_short}"',
        fontsize=8,
    )
    plt.show()

## Explore


In [ ]:
# All representatives for a few image queries
for q in range(5):
    show_i2t(q)

In [ ]:
# All representatives for a few text queries
for q in range(5):
    show_t2i(q)

In [ ]:
# Focus on a single representative to drill down
show_i2t(0, rep_indices=[0])
show_t2i(0, rep_indices=[0])

In [ ]:
# Custom condition: run combiner on-the-fly for any arbitrary vector
@torch.no_grad()
def show_custom_i2t(
    query_img_idx: int, condition: torch.Tensor, label="custom", k=PRECOMP_K
):
    """For ad-hoc conditions not in the precomputed representatives."""
    img_n = F.normalize(all_img_emb[[query_img_idx]].to(DEVICE), dim=-1)
    txt_raw = all_txt_emb.to(DEVICE)
    cond = condition.to(DEVICE).unsqueeze(0).expand(n_txt, -1)
    txt_mod = combiner(txt_raw, None, cond)
    top = torch.topk((img_n @ txt_mod.T).squeeze(0).cpu(), k=k).indices.tolist()
    is_gt = _gt_i2t(query_img_idx, top)

    clip_top = clip_i2t[query_img_idx, :k].tolist()
    columns = [
        {
            "label": "CLIP\n(baseline)",
            "top_k": clip_top,
            "is_gt": _gt_i2t(query_img_idx, clip_top),
        },
        {"label": label, "top_k": top, "is_gt": is_gt},
    ]
    img_w, col_w, row_h = 2.2, 3.0, 1.0
    fig = plt.figure(
        figsize=(img_w + col_w * len(columns), row_h * k + 0.5), constrained_layout=True
    )
    gs = gridspec.GridSpec(
        k, 1 + len(columns), figure=fig, width_ratios=[img_w] + [col_w] * len(columns)
    )
    ax0 = fig.add_subplot(gs[:, 0])
    img = _img(query_img_idx)
    (
        ax0.imshow(img)
        if img
        else ax0.text(
            0.5,
            0.5,
            f"img {query_img_idx}",
            ha="center",
            va="center",
            transform=ax0.transAxes,
            fontsize=7,
        )
    )
    ax0.axis("off")
    ax0.set_title(f"Query\nimg {query_img_idx}", fontsize=8, pad=2)
    for col, cd in enumerate(columns):
        for row in range(k):
            ax = fig.add_subplot(gs[row, col + 1])
            tidx = cd["top_k"][row]
            gt = cd["is_gt"][row]
            bg = "#b6f5b0" if gt else "white"
            ax.set_facecolor(bg)
            ax.text(
                0.5,
                0.5,
                f"#{row+1} {'✓' if gt else ''}\n"
                + "\n".join(textwrap.wrap(_short_cap(tidx), 28)),
                ha="center",
                va="center",
                fontsize=6,
                transform=ax.transAxes,
                bbox=dict(boxstyle="square,pad=0", facecolor=bg, linewidth=0),
            )
            ax.axis("off")
            if row == 0:
                ax.set_title(cd["label"], fontsize=7, pad=2)
    fig.suptitle(f"I2T custom  image {query_img_idx}  [epoch {epoch}]", fontsize=9)
    plt.show()


# Example: mean of all learned conditions
cond_mean = label_embeddings_all.mean(dim=0)
show_custom_i2t(0, cond_mean, label="global_mean")

# Example: arbitrary hand-crafted condition
show_custom_i2t(0, torch.tensor([1.0, 0.0]), label="[1, 0]")
show_custom_i2t(0, torch.tensor([-1.0, 0.0]), label="[-1, 0]")